# 🔗 تحليل دمج المستشعرات

تحليل تأثير دمج مستشعرات متعددة

**تم التطوير بمساعدة Perplexity AI**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print('✅ المكتبات جاهزة')

## 1️⃣ محاكاة دمج المستشعرات

In [ ]:
np.random.seed(42)
N = 10000

# مواقع حقيقية
true_pos = np.random.uniform(-0.03, 0.03, (N, 3))
true_pos[:, 2] = np.random.uniform(0.03, 0.07, N)

# محاكاة مستشعرات
audio_pos = true_pos + np.random.randn(N, 3) * 0.0007  # 0.7 mm
imu_pos = true_pos + np.random.randn(N, 3) * 0.0015  # 1.5 mm
camera_pos = true_pos + np.random.randn(N, 3) * 0.0010  # 1.0 mm
emg_pos = true_pos + np.random.randn(N, 3) * 0.0020  # 2.0 mm

print('✅ تم محاكاة المستشعرات')

## 2️⃣ دمج بوزنات مختلفة

In [ ]:
configs = {
    'Audio only': {'audio': 1.0, 'imu': 0.0, 'camera': 0.0, 'emg': 0.0},
    'Audio + IMU': {'audio': 0.7, 'imu': 0.3, 'camera': 0.0, 'emg': 0.0},
    'Audio + Camera': {'audio': 0.7, 'imu': 0.0, 'camera': 0.3, 'emg': 0.0},
    'Audio + IMU + Camera': {'audio': 0.6, 'imu': 0.2, 'camera': 0.2, 'emg': 0.0},
    'All sensors': {'audio': 0.6, 'imu': 0.2, 'camera': 0.15, 'emg': 0.05},
}

results = {}

for name, weights in configs.items():
    fused = (
        weights['audio'] * audio_pos +
        weights['imu'] * imu_pos +
        weights['camera'] * camera_pos +
        weights['emg'] * emg_pos
    )
    
    errors = np.linalg.norm(fused - true_pos, axis=1) * 1000  # mm
    results[name] = {
        'mean': np.mean(errors),
        'std': np.std(errors),
        'p90': np.percentile(errors, 90),
        'p95': np.percentile(errors, 95),
    }

print('=' * 70)
print('📊 نتائج دمج المستشعرات')
print('=' * 70)
for name, res in results.items():
    print(f'{name:<25}: Mean = {res["mean"]:.2f} ± {res["std"]:.2f} mm, P90 = {res["p90"]:.2f} mm')
print('=' * 70)

## 3️⃣ تصور النتائج

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

names = list(results.keys())
means = [results[name]['mean'] for name in names]
stds = [results[name]['std'] for name in names]

x = np.arange(len(names))
width = 0.6

bars = ax.bar(x, means, width, yerr=stds, capsize=5, color='skyblue')

ax.set_xlabel('تكوين المستشعرات')
ax.set_ylabel('متوسط الخطأ (ملم)')
ax.set_title('تأثير دمج المستشعرات على الدقة')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.grid(True, alpha=0.3, axis='y')

for i, m in enumerate(means):
    ax.text(i, m + stds[i] + 0.05, f'{m:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('sensor_fusion_results.png', dpi=150, bbox_inches='tight')
print('✅ Saved: sensor_fusion_results.png')
plt.show()

## 4️⃣ خلاصة

In [ ]:
print('=' * 70)
print('🎯 خلاصة دمج المستشعرات')
print('=' * 70)
best_config = min(results.keys(), key=lambda k: results[k]['mean'])
print(f'✅ أفضل تكوين: {best_config}')
print(f'✅ متوسط الخطأ: {results[best_config]["mean"]:.2f} mm')
print(f'✅ تحسن vs Audio only: {(results["Audio only"]["mean"] - results[best_config]["mean"]) / results["Audio only"]["mean"] * 100:.1f}%')
print('=' * 70)